En este ejercicio se implementará metodos como notifyall(), wait() y la clase Condition para resolverlos.

Condition es un mutex que evitará el acceso al bloque de codigo sensible y trabajará conjuntamente con notifyall() para poder manejar la seguridad de multiples recursos que idealmente se deben consultar en simultaneo, en un entorno de N hilos.

Cuando se notifica a todos los hilos, estos se despiertan inmediatamente despues de un wait() e intentan recuperar el lock sobre el Condition. El primero que lo recupera es quien puede entrar a la zona sensible.

notify_all() + while + mutex forman un conjunto estrategico.


In [ ]:
import threading
import random
import time

IDA = "IDA"
VUELTA = "VUELTA"


class Babuino:
    def __init__(self, numero):
        self.numero = numero
        self.direccion = random.choice([IDA, VUELTA])


class Cuerda:
    def __init__(self):
        self.condicion = threading.Condition()
        self.babuinos_en_cuerda = 0
        self.direccion = None

In [ ]:
def entrar_cuerda(babuino, cuerda):

    with cuerda.condicion:

        while (cuerda.babuinos_en_cuerda >= 5 or
               (cuerda.direccion is not None and
                cuerda.direccion != babuino.direccion)):    # Solo hay acceso mientras no haya direccion establecida o esta es la misma del babuino que quiere entrar

            cuerda.condicion.wait()

        cuerda.babuinos_en_cuerda += 1
        cuerda.direccion = babuino.direccion

        print(
            f"Babuino {babuino.numero} entra a la cuerda "
            f"({babuino.direccion}). "
            f"Hay {cuerda.babuinos_en_cuerda} babuinos."
        )

El with debería durar solamente lo necesario para consultar o modificar el estado compartido. El tiempo de cruce no modifica ese estado, y puede tardar bastante, afectando innecesariamente el flujo del programa.

Por otro lado, en entraR_cuerda(..) no es necesario el notify_All() aun, porque los hilos bloqueados lo están por una de las dos siguientes razones:
1. La cuerda estaba llena       → cantidad >= 5
2. La dirección era contraria   → dirección != mi_dirección

Osea que recien al salir de la cuerda un babuino es el único escenario donde los hilos bloqueados pueden volver a preguntar.

 Esto no evita que haya nuevos hilos que sigan preguntando por el condition de entrar_cuerda() una vez que el último hilo salió de esta **funcion** (porque ahi, el "with" ejecuta automaticamente un release() del condition).

In [ ]:
def cruzar_cuerda(babuino):
    print(f"Babuino {babuino.numero} cruzando hacia {babuino.direccion}")

    tiempo = random.uniform(1, 3)
    time.sleep(tiempo)

In [ ]:
def salir_cuerda(babuino, cuerda):

    with cuerda.condicion:

        cuerda.babuinos_en_cuerda -= 1

        print(
            f"Babuino {babuino.numero} salió de la cuerda. "
            f"Quedan {cuerda.babuinos_en_cuerda} babuinos."
        )

        if cuerda.babuinos_en_cuerda == 0:
            cuerda.direccion = None

        cuerda.condicion.notify_all()

In [ ]:
def trabajar(babuino, cuerda):

    entrar_cuerda(babuino, cuerda)

    cruzar_cuerda(babuino)

    salir_cuerda(babuino, cuerda)

In [ ]:
def iniciar_simulacion(n):
    cuerda = Cuerda()
    hilos = []

    for i in range(n):
        babuino = Babuino(i + 1)

        hilo = threading.Thread(
            target=trabajar,
            args=(babuino, cuerda)
        )

        hilos.append(hilo)
        hilo.start()

    for hilo in hilos:
        hilo.join()

    print("\nTodos los babuinos terminaron de cruzar.")

In [ ]:
iniciar_simulacion(10)

Babuino 1 entra a la cuerda (VUELTA). Hay 1 babuinos.
Babuino 1 cruzando hacia VUELTA
Babuino 3 entra a la cuerda (VUELTA). Hay 2 babuinos.
Babuino 3 cruzando hacia VUELTA
Babuino 4 entra a la cuerda (VUELTA). Hay 3 babuinos.
Babuino 4 cruzando hacia VUELTA
Babuino 6 entra a la cuerda (VUELTA). Hay 4 babuinos.
Babuino 6 cruzando hacia VUELTA
Babuino 6 salió de la cuerda. Quedan 3 babuinos.
Babuino 4 salió de la cuerda. Quedan 2 babuinos.
Babuino 1 salió de la cuerda. Quedan 1 babuinos.
Babuino 3 salió de la cuerda. Quedan 0 babuinos.
Babuino 5 entra a la cuerda (IDA). Hay 1 babuinos.
Babuino 5 cruzando hacia IDA
Babuino 2 entra a la cuerda (IDA). Hay 2 babuinos.
Babuino 2 cruzando hacia IDA
Babuino 9 entra a la cuerda (IDA). Hay 3 babuinos.
Babuino 9 cruzando hacia IDA
Babuino 8 entra a la cuerda (IDA). Hay 4 babuinos.
Babuino 8 cruzando hacia IDA
Babuino 10 entra a la cuerda (IDA). Hay 5 babuinos.
Babuino 10 cruzando hacia IDA
Babuino 9 salió de la cuerda. Quedan 4 babuinos.
Babuino 7

El orden de quien entra primero, sale primero se respeta. No importa si numericamente no pareciere tener sentido (por ejemplo, entra babuino 10 antes que babuino 9), porque eso depende de quien agarró el mutex primero despues de un notify all, o de si el babuino 10 quizas si iba en la direccion actual de la cuerda, mientras que babuino 9 no, haciendo que este ultimo, si bien preguntó primero por el recurso, tenga que esperar.